# 51 — Profile Agent
Runs `44-Dataframe_comet.py` (player 0) vs a random agent (player 1) for up to 100 steps.
Measures wall-clock time per step and uses `line_profiler` to show per-line hotspots.

In [1]:
import importlib.util
import time
import random
import math

import kaggle_environments as ke
import plotly.graph_objects as go
from line_profiler import LineProfiler

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 17.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_goofspiel
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hearts
[kaggle_environments.envs.open_sp

In [2]:
# Load 44-Dataframe_comet.py as a fresh module — resets global step/player state
spec = importlib.util.spec_from_file_location("agent57", "57-Dataframe_comet_numba.py")
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)
m.step = 0
m.num_agents = None
m.player_id = None

In [3]:
SEED = 42
N_STEPS = 100
random.seed(SEED)

def random_agent_fn(obs):
    player = obs.player
    my_planets = [p for p in obs.planets if p[1] == player]
    if not my_planets:
        return []
    planet = random.choice(my_planets)
    ships = planet[5] // 2
    if ships < 1:
        return []
    return [[planet[0], random.uniform(0, 2 * math.pi), ships]]

In [4]:
# Instrument the four key functions with line_profiler
lp = LineProfiler()
lp.add_function(m._simulate)
lp.add_function(m.take_action)
lp.add_function(m.IntervalProcessor.create_cumulative_obstacles)
profiled_agent = lp(m.nearest_planet_sniper)

In [5]:
env = ke.make("orbit_wars", debug=False)
env.reset(2)

step_numbers: list[int] = []
step_times_ms: list[float] = []

for env_step in range(N_STEPS):
    obs0 = env.state[0].observation
    obs1 = env.state[1].observation

    t0 = time.perf_counter()
    action0 = profiled_agent(obs0)
    dt_ms = (time.perf_counter() - t0) * 1000
    step_numbers.append(env_step)
    step_times_ms.append(dt_ms)
    print(f"Step {env_step:3d}: {dt_ms:7.2f} ms")

    action1 = random_agent_fn(obs1)
    env.step([action0, action1])
    if env.state[0].status != "ACTIVE":
        break

Agent called step: 0 remainingOverageTime: 60
Step   0: 11560.30 ms
Agent called step: 1 remainingOverageTime: 60
Step   1:  442.91 ms
Agent called step: 2 remainingOverageTime: 60
Step   2:  414.96 ms
Agent called step: 3 remainingOverageTime: 60
From 4, To 20 at step 10 with 13 ships (target has min 13)
Step   3:  325.91 ms
Agent called step: 4 remainingOverageTime: 60
Step   4:  388.26 ms
Agent called step: 5 remainingOverageTime: 60
Step   5:  304.24 ms
Agent called step: 6 remainingOverageTime: 60
Step   6:  422.94 ms
Agent called step: 7 remainingOverageTime: 60
Step   7:  398.67 ms
Agent called step: 8 remainingOverageTime: 60
Step   8:  198.35 ms
Agent called step: 9 remainingOverageTime: 60
Step   9:  339.90 ms
Agent called step: 10 remainingOverageTime: 60
Step  10:  413.98 ms
Agent called step: 11 remainingOverageTime: 60
Step  11:  454.67 ms
Agent called step: 12 remainingOverageTime: 60
Step  12:  383.80 ms
Agent called step: 13 remainingOverageTime: 60
From 4, To 14 at st

In [6]:
obs0 = env.state[0].observation
p0 = sum(p[5] for p in obs0.planets if p[1] == 0)
p1 = sum(p[5] for p in obs0.planets if p[1] == 1)
n = len(step_numbers)
mean_ms = sum(step_times_ms) / n

print(f"Player 0 (our agent): {p0} ships")
print(f"Player 1 (random):    {p1} ships")
winner = "Our agent wins" if p0 > p1 else "Random wins" if p1 > p0 else "Tie"
print(f"Result after {n} steps: {winner}")
print(f"Step times — min: {min(step_times_ms):.1f} ms  max: {max(step_times_ms):.1f} ms  mean: {mean_ms:.1f} ms")

Player 0 (our agent): 2672 ships
Player 1 (random):    0 ships
Result after 100 steps: Our agent wins
Step times — min: 198.4 ms  max: 11560.3 ms  mean: 863.6 ms


In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=step_numbers,
    y=step_times_ms,
    mode="lines+markers",
    name="Agent call time",
    line=dict(color="royalblue", width=1.5),
    marker=dict(size=5),
))
fig.add_hline(
    y=mean_ms,
    line_dash="dash",
    line_color="orange",
    annotation_text=f"Mean {mean_ms:.1f} ms",
    annotation_position="top right",
)
fig.update_layout(
    title="44-Dataframe_comet agent call time per step (vs random)",
    xaxis_title="Step",
    yaxis_title="Time (ms)",
    template="plotly_white",
    hovermode="x unified",
)
fig.show()

In [8]:
import io, sys

buf = io.StringIO()
lp.print_stats(stream=buf, output_unit=1e-3)
print(buf.getvalue())

Timer unit: 0.001 s

Total time: 7.1414 s
File: c:\Users\trant\Documents\Programmation\Orbit Wars\57-Dataframe_comet_numba.py
Function: _simulate at line 294

Line #      Hits         Time  Per Hit   % Time  Line Contents
   294                                           def _simulate(obs, global_step, num_agents, n_steps=NB_STEPS_SIM):
   295       100        630.9      6.3      8.8      sim = copy.deepcopy(obs)
   296       100          0.4      0.0      0.0      no_actions = [[] for _ in range(num_agents)]
   297       100          0.1      0.0      0.0      rows = []
   298      1200          1.1      0.0      0.0      for i in range(n_steps+1):
   299     37620         16.4      0.0      0.2          for p in sim.planets:
   300     36520         21.5      0.0      0.3              pid, owner, x, y, radius, ships, production = (
   301     36520         19.2      0.0      0.3                  p[0], p[1], p[2], p[3], p[4], p[5], p[6]
   302                                           